In [0]:
dbutils.widgets.text("catalog", "pa_clock_dev", "Catalog")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"Create CATALOG if not exists `{catalog}`")

spark.sql(f"USE CATALOG `{catalog}`")

# Schemas: ref (static data) + medallion layers
for s in ["ref", "bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{s}`")

# 1) PA decision-window rules (CMS-0057-F: 72h expedited, 7 calendar days = 168h standard)
spark.sql("""
CREATE OR REPLACE TABLE ref.ref_pa_rules AS
SELECT * FROM VALUES
  ('MA',       'STANDARD',  168, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE)),
  ('MA',       'EXPEDITED',  72, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE)),
  ('MEDICAID', 'STANDARD',  168, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE)),
  ('MEDICAID', 'EXPEDITED',  72, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE)),
  ('CHIP',     'STANDARD',  168, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE)),
  ('CHIP',     'EXPEDITED',  72, 0.75, 0.90, DATE'2026-01-01', CAST(NULL AS DATE))
AS t(line_of_business, urgency, hours_allowed, warn_pct, critical_pct, effective_from, effective_to)
""")

# 2) 200 synthetic members (deterministic: 100 MA, 60 MEDICAID, 40 CHIP)
spark.sql("""
CREATE OR REPLACE TABLE ref.dim_member AS
SELECT
  concat('M', lpad(CAST(id AS STRING), 5, '0')) AS member_id,
  CASE WHEN id % 10 < 5 THEN 'MA'
       WHEN id % 10 < 8 THEN 'MEDICAID'
       ELSE 'CHIP' END AS line_of_business,
  element_at(array('NJ','NY','PA','TX','FL'), CAST(id % 5 AS INT) + 1) AS state
FROM range(1, 201)
""")

# Quick check
display(spark.table("ref.ref_pa_rules"))
display(spark.sql("SELECT line_of_business, count(*) AS members FROM ref.dim_member GROUP BY 1 ORDER BY 1"))

In [0]:
%sql
--Select * from pa_clock_dev.ref.dim_member
Select * From pa_clock_dev.ref.ref_pa_rules